In [ ]:
import os
import glob
import utils
import numpy as np
import importlib

importlib.reload(utils)
import xarray as xr

print("starting script", flush=True)
base_path = "../data/rsds_past"

files_past = []
files_future = []

for root, dirs, files in os.walk(base_path):
    if not dirs:
        rel_path = os.path.relpath(root, base_path)
        path_past = os.path.join(base_path, rel_path)
        future_path = path_past.replace("rsds_past", "rsds_future")

        # Find nc files
        past_files = glob.glob(os.path.join(path_past, "*r1i1p1_1995*.nc"))
        future_files = glob.glob(os.path.join(future_path, "*r1i1p1_2045*.nc"))

        files_past.extend(past_files)
        files_future.extend(future_files)


# Use the file names as model identifiers
model_names_past = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_past
]
model_names_future = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_future
]

print("File source loaded", flush=True)
# Process and select time frame
files_raw_past = utils.pre_process(files_past, ["rsds"], 0, 5, 0, 5)
files_past = utils.select_time_frame(files_raw_past, slice("1995-01-01", "2004-12-30"))

print("Files loaded", flush=True)

In [ ]:
marginal = utils.marginal_average_seasonal_hourly(files_past)


seasons = ["DJF", "MAM", "JJA", "SON"]

# stack all models
model_arrays = []
for m in range(len(marginal)):
    season_dict = marginal[m]
    combined = xr.concat(
        [season_dict[s] for s in seasons], dim=xr.Variable("season", seasons)
    )
    model_arrays.append(combined)

# concatenate along  model dimension
all_models_array = xr.concat(
    model_arrays, dim=xr.Variable("model", range(len(marginal)))
)

seasonal_mean = all_models_array.values
np.save(
    "../plotting_data/rsds_spatial_raw/marginal_average_seasonal_hourly.npy",
    seasonal_mean,
)

In [ ]:
marginal = utils.marginal_var_seasonal(files_past)


seasons = ["DJF", "MAM", "JJA", "SON"]
# stack all models
model_arrays = []
for m in range(len(marginal)):
    season_dict = marginal[m]
    combined = xr.concat(
        [season_dict[s] for s in seasons], dim=xr.Variable("season", seasons)
    )
    model_arrays.append(combined)

# concatenate along  model dimension
all_models_array = xr.concat(
    model_arrays, dim=xr.Variable("model", range(len(marginal)))
)

# numpy array: shape (model, season, lat, lon)
seasonal_var = all_models_array.values
np.save(
    "../plotting_data/rsds_spatial_raw/marginal_var_seasonal_hourly.npy",
    seasonal_var,
)

In [ ]:
marginal = utils.spatiotemporal_below_percentile_seasonal(files_past)


seasons = ["DJF", "MAM", "JJA", "SON"]
# stack all models
model_arrays = []
for m in range(len(marginal)):
    season_dict = marginal[m]
    combined = xr.concat(
        [season_dict[s] for s in seasons], dim=xr.Variable("season", seasons)
    )
    model_arrays.append(combined)

# concatenate along  model dimension
all_models_array = xr.concat(
    model_arrays, dim=xr.Variable("model", range(len(marginal)))
)

# numpy array: shape (model, season, lat, lon)
low_percentile = all_models_array.values
np.save(
    "../plotting_data/rsds_spatial_raw/90th_percentile.npy",
    low_percentile,
)

In [ ]:
marginal = utils.spatiotemporal_below_percentile_seasonal(files_past, percentile=50)


seasons = ["DJF", "MAM", "JJA", "SON"]
# stack all models
model_arrays = []
for m in range(len(marginal)):
    season_dict = marginal[m]
    combined = xr.concat(
        [season_dict[s] for s in seasons], dim=xr.Variable("season", seasons)
    )
    model_arrays.append(combined)

# concatenate along  model dimension
all_models_array = xr.concat(
    model_arrays, dim=xr.Variable("model", range(len(marginal)))
)

# numpy array: shape (model, season, lat, lon)
median_percentile = all_models_array.values
np.save(
    "../plotting_data/rsds_spatial_raw/50th_percentile.npy",
    median_percentile,
)

In [ ]:
files_past = []
files_future = []

for root, dirs, files in os.walk(base_path):
    if not dirs:  # lowest-level folder
        rel_path = os.path.relpath(root, base_path)
        path_past = os.path.join(base_path, rel_path)
        future_path = path_past.replace("rsds_past", "rsds_future")

        # Find nc files
        past_files = glob.glob(os.path.join(path_past, "*r1i1p1_1995*.nc"))
        future_files = glob.glob(os.path.join(future_path, "*r1i1p1_2045*.nc"))

        files_past.extend(past_files)
        files_future.extend(future_files)


# Use the file names as model identifiers
model_names_past = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_past
]
model_names_future = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_future
]


files_raw_past = utils.pre_process(files_past[0:1], ["rsds"], 0, 5, 0, 5)
print(files_raw_past)
files_past = utils.select_time_frame(files_raw_past, slice("1995-01-01", "2004-12-30"))

# Compute relative variability (RCM / GCM)

# Coordinates
lons = files_past[0].coords["lon"].values
lats = files_past[0].coords["lat"].values

import os

# Define and create the directory
output_dir = "../plotting_data/rsds_spatial_raw/"
os.makedirs(output_dir, exist_ok=True)

# 1. Save Lons and Lats (as plain text, one value per line)
np.savetxt(os.path.join(output_dir, "lons.txt"), lons, delimiter="\n")
np.savetxt(os.path.join(output_dir, "lats.txt"), lats, delimiter="\n")

# 2. Save Model Names (one name per line)
with open(os.path.join(output_dir, "model_names_past.txt"), "w") as f:
    f.write("\n".join(model_names_past))

with open(os.path.join(output_dir, "model_names_future.txt"), "w") as f:
    f.write("\n".join(model_names_future))

print(f"Files saved successfully in {output_dir}")